# 관계 파악 차트 — 산점도 · 상관계수 · 히트맵


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 산점도로 두 수치형 변수의 관계 방향과 강도를 파악하고, 피어슨 상관계수(`corr()`)와 히트맵(`sns.heatmap`), 주변 분포 결합 차트(`jointplot`, `pairplot`)를 작성하며 상관관계와 인과관계를 구분합니다.

## 처음 배우는 분을 위한 안내

산점도는 한 관측을 점 하나로 그립니다. X축·Y축에 비교할 숫자 열을 놓습니다. 상관계수는 직선 관계의 방향·강도를 −1 ~ 1로 요약하며 인과를 입증하지 않습니다. hue는 그룹별 색상, 히트맵은 값을 색으로 나타내는 표, pairplot은 여러 변수 쌍의 산점도 모음입니다.

**학습 순서:** 문제 1 ~ 4는 필수 기초, 여러 변수를 묶는 5 ~ 12는 선택 도전입니다. 문제 2의 online_clean은 이후 문제 6·7·10에서도 씁니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 환경과 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요.

```python
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 고객별 주문 요약 가상 데이터 (10명)
summary = pd.DataFrame({
    'customer_id': [f'C{i:02d}' for i in range(1, 11)],
    'total_orders': [1, 2, 2, 3, 4, 5, 6, 7, 8, 10],
    'total_spent': [50, 90, 110, 140, 210, 240, 310, 350, 420, 520],
    'avg_order_value': [50.0, 45.0, 55.0, 46.67, 52.5, 48.0, 51.67, 50.0, 52.5, 52.0],
    'tier': ['Bronze', 'Bronze', 'Bronze', 'Silver', 'Silver',
             'Silver', 'Gold', 'Gold', 'Gold', 'Gold']
})

# 온라인 주문 가상 데이터 (결측치 1건 포함, 12행)
online = pd.DataFrame({
    'order_id': range(101, 113),
    'region': ['South', 'North', 'South', 'East', 'North', 'South',
               'East', 'North', 'South', 'East', 'North', 'South'],
    'quantity': [1, 2, 2, 3, 1, 4, 2, 3, 5, 2, 4, 3],
    'unit_price': [20.0, 30.0, 25.0, 40.0, None, 50.0, 35.0, 60.0, 45.0, 80.0, 70.0, 90.0],
    'discount': [0.00, 0.05, 0.00, 0.10, 0.05, 0.15, 0.05, 0.10, 0.20, 0.05, 0.15, 0.10],
    'delivery_days': [2, 3, 2, 5, 4, 6, 3, 7, 8, 4, 9, 5],
    'customer_rating': [4.8, 4.5, 4.7, 3.8, 4.2, 3.5, 4.4, 3.2, 3.0, 4.1, 2.8, 3.9],
    'revenue': [20.0, 57.0, 50.0, 108.0, None, 170.0, 66.5, 162.0, 180.0, 152.0, 238.0, 243.0]
})

# 연도·월별 주가/매출 피벗 히트맵용 데이터
monthly_perf = pd.DataFrame({
    'year': [2024, 2024, 2024, 2024, 2025, 2025, 2025, 2025],
    'month': [1, 2, 3, 4, 1, 2, 3, 4],
    'close_avg': [110.0, 140.0, 135.0, 160.0, 210.0, 240.0, 255.0, 280.0]
})
```

## 산점도(Scatter Plot)와 패턴 해석

산점도는 두 수치형 변수를 각각 X축과 Y축에 놓고 관측값 하나를 점 하나로 표시하는 차트입니다. 데이터가 많을 때는 `alpha=0.6`처럼 투명도를 주어 점이 겹치는 밀집 구간을 구분하고, 특정 데이터 포인트를 강조할 때는 `plt.annotate()`로 라벨을 붙입니다.

| 점들의 분포 패턴 | 관계 방향과 강도 | 해석 예시 |
| --- | --- | --- |
| 왼쪽 아래 → 오른쪽 위 | 양(+)의 상관관계 | 광고비·주문 수가 늘수록 매출도 함께 증가 |
| 왼쪽 위 → 오른쪽 아래 | 음(-)의 상관관계 | 배송 지연 일수가 길어질수록 고객 평점 감소 |
| 직선 주변에 좁게 밀집 | 강한 선형 관계 | 한 변수로 다른 변수의 범위를 좁게 예측 가능 |
| 넓고 흐릿하게 퍼짐 | 약한 선형 관계 | 방향성은 있으나 다른 요인의 영향이 큼 |
| 무작위 구름 형태 / 곡선 | 선형 무상관 (또는 비선형 관계) | 상관계수가 0에 가까워도 U자형 비선형 패턴인지 산점도로 확인 필요 |

```python
plt.figure(figsize=(6, 4))
plt.scatter(summary['total_orders'], summary['total_spent'], color='steelblue', alpha=0.8)
for _, row in summary.iloc[[0, -1]].iterrows():
    plt.annotate(
        row['customer_id'],
        xy=(row['total_orders'], row['total_spent']),
        xytext=(5, -5),
        textcoords='offset points'
    )
plt.xlabel('총 주문 횟수')
plt.ylabel('총 구매 금액')
plt.title('주문 횟수와 총 구매 금액의 산점도')
plt.grid(True, alpha=0.3)
plt.show()
```

## 피어슨 상관계수(`r`)의 원리와 해석

산점도로 눈으로 확인한 선형 관계를 `-1`에서 `+1` 사이의 숫자로 요약한 값이 **피어슨 상관계수(Pearson correlation coefficient, `r`)**입니다.

- **표준편차(Standard Deviation):** 각 변수가 자신의 평균으로부터 얼마나 퍼져 있는지 나타냅니다.
- **공분산(Covariance):** 두 변수가 함께 변하는 방향을 나타내지만, 변수 단위(원, 만 원, 건수)에 따라 값 크기가 달라져 강도를 직접 비교하기 어렵습니다.
- **정규화(Normalization):** 공분산을 두 변수의 표준편차 곱으로 나누어 단위 영향을 제거한 것이 피어슨 상관계수(`-1 ≤ r ≤ +1`)입니다.

| 상관계수(`r`) 범위 | 선형 관계 해석 |
| --- | --- |
| `0.7 ≤ |r| ≤ 1.0` | 강한 선형 상관관계 (`r > 0`이면 양, `r < 0`이면 음) |
| `0.3 ≤ |r| < 0.7` | 중간 정도의 선형 상관관계 |
| `0.0 ≤ |r| < 0.3` | 약한 선형 상관관계 (또는 거의 없음) |

```python
r_val = summary['total_orders'].corr(summary['total_spent'])
print(f"주문 횟수와 총 구매금액 상관계수: {r_val:.4f}")
```

## 상관관계 히트맵과 피벗 집계 히트맵

수치형 변수가 여러 개일 때는 `df[수치열_목록].corr()`로 상관계수 행렬을 구한 뒤 `sns.heatmap()`으로 한눈에 비교합니다.

- **상관계수 히트맵 핵심 옵션:** `annot=True`(칸 안에 숫자 표시), `fmt='.2f'`(소수점 둘째 자리), `cmap='coolwarm'`, `vmin=-1, vmax=1, center=0`(색상 기준을 `-1 ~ +1`, 중심 `0`에 고정).
- 대각선(`(i, i)`)은 값이 변하는 열이고 유효 관측이 충분하면 **자기 자신과의 상관계수**이므로 `1.00`입니다.
- **범주×범주 집계 히트맵:** 상관계수뿐만 아니라 `groupby` + `pivot_table`로 만든 교차 집계표(예: 연도×월 평균 주가, 탑승항구×객실등급 승객 수)도 `sns.heatmap()`으로 시각화하면 어느 조합에 값이 집중되는지 즉시 파악할 수 있습니다.

```python
num_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']
corr_mat = online[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_mat, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('주문 데이터 수치형 변수 상관관계 히트맵')
plt.tight_layout()
plt.show()
```

## 확장 관계 차트: `scatterplot(hue=...)`, `jointplot`, `pairplot`

Seaborn은 산점도에 범주 구분과 단변량 분포를 결합한 고수준 함수를 제공합니다.

1. **`sns.scatterplot(data=..., x=..., y=..., hue='범주열')`**: 그룹별 색상을 다르게 찍어 전체 추세와 하위 그룹별 클러스터를 동시에 보여줍니다.
2. **`sns.jointplot(data=..., x=..., y=..., hue=...)`**: 중앙에는 산점도를, 상단과 우측 여백에는 각 변수의 히스토그램(또는 KDE)을 함께 배치해 **두 변수의 개별 분포와 결합 관계**를 한 번에 보여줍니다.
3. **`sns.pairplot(df, hue=...)`**: 데이터프레임 내 모든 수치형 변수 쌍의 산점도와 대각선 단변량 분포를 격자로 그립니다. 변수 수가 많거나 행 수가 클 때는 실행 시간이 오래 걸리므로 핵심 열만 골라 실행하는 것이 좋습니다.

## 상관관계 ≠ 인과관계 (교란 변수 주의)

상관계수가 높게 나왔다고 해서 **"X를 바꾸면 Y가 바뀐다(인과관계)"**고 결론 내리면 안 됩니다.

- **교란 변수(Confounding Variable, 제3의 요인):** 아이스크림 판매량과 익사 사고 건수는 양의 상관관계가 높지만, 아이스크림이 사고의 원인이 아니라 **'여름철 기온 상승'**이라는 제3의 변수가 양쪽 모두를 증가시킨 것입니다.
- 쇼핑몰 데이터에서 상품 단가(`unit_price`)와 매출(`revenue`)의 상관계수가 높더라도, 무작정 가격을 올리면 구매 수량(`quantity`)이 급감하거나 이탈이 발생할 수 있습니다. 프로모션 여부(`discount`), 상품 카테고리, 구매 수량 등 함께 작동하는 변수를 반드시 함께 점검해야 합니다.

## 선택 도전: 네 점의 선형 관계 확인

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_pairs = pd.DataFrame({'hours':[1,2,3,4],'score':[2,4,6,8]})
```

### 진행 순서

1. hours를 X축, score를 Y축으로 산점도를 그립니다.
2. 두 열의 corr()를 출력합니다.
3. 이 가상 규칙을 실제 공부시간의 인과 효과라고 주장할 수 없는 이유를 한 문장으로 적습니다.

**완료 기준:** 점 4개가 오른쪽 위로 향하는 직선 위에 있고 상관계수 1.0. 실제 현상을 검증한 자료는 아님.

### 선택 도전 풀이

```python
import matplotlib.pyplot as plt
plt.scatter(extra_pairs['hours'],extra_pairs['score'])
plt.xlabel('Hours');plt.ylabel('Score');plt.title('Four Pairs');plt.show()
extra_r=extra_pairs['hours'].corr(extra_pairs['score']);print(extra_r)
assert abs(extra_r-1)<1e-10
```

## 복습 순서

1. 두 수치형 변수의 관계를 볼 때 상관계수만 구하지 않고 산점도를 먼저 그려 이상치나 비선형 패턴을 확인합니다.
2. `corr()`의 부호(방향)와 절댓값 크기(강도)를 해석 기준표와 대조합니다.
3. `sns.heatmap`에서 `vmin=-1, vmax=1, center=0`을 설정하는 이유와 대각선 `1.00`의 의미를 설명합니다.
4. 상관관계를 인과관계로 오해하기 쉬운 사례에서 숨은 제3의 변수(교란 변수)를 찾아봅니다.